# F1 Podium Prediction — Milestone 1

Goal: for each driver in each race, predict whether they finish on the podium
(`results.positionOrder <= 3`), using only information available after qualifying.

This notebook currently covers **Setup**, **Data Loading** and the **Raw Data Audit**.
The audit only *measures* the raw data; it does not clean or decide anything.
Reusable code lives in `src/f1podium/`.

## 1. Setup

Make the `src/` folder importable. We search upwards from the current folder so the
same cell works when run from `notebooks/`, from the repo root, or on Kaggle
(where `src/` must be provided alongside the notebook).

In [1]:
import sys
from pathlib import Path

def _add_src_to_path() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "f1podium").is_dir():
            src = base / "src"
            if str(src) not in sys.path:
                sys.path.insert(0, str(src))
            return src
    raise FileNotFoundError("Could not find src/f1podium above " + str(Path.cwd()))

SRC_DIR = _add_src_to_path()

%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from f1podium import config
from f1podium.load import load_all
from f1podium.validate import (
    find_duplicates,
    missing_summary,
    run_foreign_key_checks,
    run_primary_key_checks,
)

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

np.random.seed(config.RANDOM_SEED)

print("Python     :", sys.version.split()[0], "|", sys.executable)
print("pandas     :", pd.__version__)
print("src        :", SRC_DIR)
print("raw data   :", config.RAW_DIR)
print("split      : train <=", config.TRAIN_END_YEAR, "| val", config.VAL_YEARS,
      "| test >=", config.TEST_START_YEAR)
print("seed       :", config.RANDOM_SEED)

Python     : 3.13.7 | C:\Users\yshim\dev\f1-podium\.venv\Scripts\python.exe
pandas     : 3.0.6
src        : C:\Users\yshim\dev\f1-podium\src
raw data   : C:\Users\yshim\dev\f1-podium\data\raw
split      : train <= 2019 | val (2020, 2021) | test >= 2022
seed       : 42


## 2. Data Loading

`load_all()` reads the 14 CSVs with an explicit schema per table:
`\N` (and empty fields) become missing; IDs/counts use nullable `Int64`;
text uses `string`; dates are parsed strictly with `%Y-%m-%d`.
The loader raises if a file's columns differ from the declared schema.

In [3]:
tables = load_all()

load_summary = pd.DataFrame(
    {
        "rows": {n: len(df) for n, df in tables.items()},
        "columns": {n: df.shape[1] for n, df in tables.items()},
        "memory_MB": {n: round(df.memory_usage(deep=True).sum() / 1e6, 2) for n, df in tables.items()},
    }
).rename_axis("table")
display(load_summary)

,rows,columns,memory_MB
table,,,
circuits,77,9,0.03
constructor_results,12625,5,1.15
constructor_standings,13391,7,1.38
constructors,212,5,0.06
driver_standings,34863,7,3.62
drivers,861,9,0.34
lap_times,589081,6,60.09
pit_stops,11371,7,1.79
qualifying,10494,9,2.35


Declared dtypes of the three tables we will use most:

In [4]:
for name in ["races", "results", "qualifying"]:
    print(f"--- {name}")
    print(tables[name].dtypes.to_string(), end="\n\n")

--- races
raceId                  Int64
year                    Int64
round                   Int64
circuitId               Int64
name                   string
date           datetime64[us]
time                   string
url                    string
fp1_date       datetime64[us]
fp1_time               string
fp2_date       datetime64[us]
fp2_time               string
fp3_date       datetime64[us]
fp3_time               string
quali_date     datetime64[us]
quali_time             string
sprint_date    datetime64[us]
sprint_time            string

--- results
resultId             Int64
raceId               Int64
driverId             Int64
constructorId        Int64
number               Int64
grid                 Int64
position             Int64
positionText        string
positionOrder        Int64
points             float64
laps                 Int64
time                string
milliseconds         Int64
fastestLap           Int64
rank                 Int64
fastestLapTime      string
faste

In [5]:
# Short aliases used throughout the audit
races = tables["races"]
results = tables["results"]
qualifying = tables["qualifying"]

def with_year(df: pd.DataFrame) -> pd.DataFrame:
    """Attach races.year, checking the join neither drops nor duplicates rows."""
    out = df.merge(races[["raceId", "year"]], on="raceId", how="left", validate="many_to_one")
    assert len(out) == len(df), "join changed the row count"
    assert out["year"].notna().all(), "some rows have a raceId not in races"
    return out

## 3. Raw Data Audit

### 3.1 Missing values per column

Count and percentage of missing values for every column of every table
(after treating `\N` and empty fields as missing).

In [6]:
missing = missing_summary(tables)
display(missing.sort_values(["table", "pct_missing"], ascending=[True, False]).reset_index(drop=True))

,table,column,dtype,n_rows,n_missing,pct_missing
0,circuits,circuitId,Int64,77,0,0.00
1,circuits,circuitRef,string,77,0,0.00
2,circuits,name,string,77,0,0.00
3,circuits,location,string,77,0,0.00
4,circuits,country,string,77,0,0.00
5,circuits,lat,float64,77,0,0.00
6,circuits,lng,float64,77,0,0.00
7,circuits,alt,Int64,77,0,0.00
8,circuits,url,string,77,0,0.00
9,constructor_results,status,string,12625,12608,99.87


Only the columns that have at least one missing value, worst first:

In [7]:
display(
    missing[missing["n_missing"] > 0]
    .sort_values("pct_missing", ascending=False)
    .reset_index(drop=True)
)

,table,column,dtype,n_rows,n_missing,pct_missing
0,constructor_results,status,string,12625,12608,99.87
1,races,sprint_time,string,1125,1110,98.67
2,races,sprint_date,datetime64[us],1125,1107,98.40
3,races,fp3_time,string,1125,1072,95.29
4,races,fp1_time,string,1125,1057,93.96
5,races,fp2_time,string,1125,1057,93.96
6,races,quali_time,string,1125,1057,93.96
7,races,fp3_date,datetime64[us],1125,1053,93.60
8,drivers,number,Int64,861,802,93.15
9,races,fp1_date,datetime64[us],1125,1035,92.00


### 3.2 Primary keys

A key passes if it has no nulls and no repeated values. Surrogate ids (e.g. `raceId`)
are listed first, then natural/composite keys, which say more about the data.
`results (raceId, driverId)` is **expected to fail**: early seasons have shared drives.

In [8]:
pk_results = run_primary_key_checks(tables)
display(pk_results)
print(f"{(~pk_results['passed']).sum()} of {len(pk_results)} primary-key checks failed")

,table,key,n_rows,n_null_key,n_duplicate_rows,passed
0,circuits,circuitId,77,0,0,True
1,constructors,constructorId,212,0,0,True
2,drivers,driverId,861,0,0,True
3,races,raceId,1125,0,0,True
4,status,statusId,139,0,0,True
5,seasons,year,75,0,0,True
6,results,resultId,26759,0,0,True
7,sprint_results,resultId,360,0,0,True
8,qualifying,qualifyId,10494,0,0,True
9,driver_standings,driverStandingsId,34863,0,0,True


1 of 24 primary-key checks failed


### 3.3 Foreign keys

For each relationship: how many child rows have a null key, and how many point to an
id that does not exist in the parent table.

In [9]:
fk_results = run_foreign_key_checks(tables)
display(fk_results)
print(f"{(~fk_results['passed']).sum()} of {len(fk_results)} foreign-key checks failed")

,child,fk,parent,n_child_rows,n_null_fk,n_unmatched_rows,n_unmatched_distinct,example_unmatched,passed
0,races,circuitId,circuits.circuitId,1125,0,0,0,[],True
1,races,year,seasons.year,1125,0,0,0,[],True
2,results,raceId,races.raceId,26759,0,0,0,[],True
3,results,driverId,drivers.driverId,26759,0,0,0,[],True
4,results,constructorId,constructors.constructorId,26759,0,0,0,[],True
5,sprint_results,raceId,races.raceId,360,0,0,0,[],True
6,sprint_results,driverId,drivers.driverId,360,0,0,0,[],True
7,sprint_results,constructorId,constructors.constructorId,360,0,0,0,[],True
8,qualifying,raceId,races.raceId,10494,0,0,0,[],True
9,qualifying,driverId,drivers.driverId,10494,0,0,0,[],True


0 of 23 foreign-key checks failed


### 3.4 Duplicate `(raceId, driverId)` rows in `results`, by year

Our modelling grain is one row per `(raceId, driverId)`. Here we count, per season,
the rows that violate it. Nothing is dropped; the rule for handling them is an open decision.

In [10]:
dups = with_year(find_duplicates(results, ["raceId", "driverId"]))

dup_by_year = pd.DataFrame({
    "n_duplicated_pairs": dups.drop_duplicates(["raceId", "driverId"]).groupby("year").size(),
    "n_rows_involved": dups.groupby("year").size(),
    "n_races_affected": dups.groupby("year")["raceId"].nunique(),
})
dup_by_year.loc["Total"] = dup_by_year.sum()
display(dup_by_year)

,n_duplicated_pairs,n_rows_involved,n_races_affected
year,,,
1950,4,8,3
1951,4,8,2
1952,3,6,3
1953,14,30,5
1954,16,33,5
1955,15,33,4
1956,12,24,6
1957,8,16,5
1958,3,6,3


Example duplicated entries (same driver, same race, more than one row):

In [11]:
example = dups.merge(
    tables["drivers"][["driverId", "driverRef"]], on="driverId", how="left", validate="many_to_one"
).merge(
    tables["constructors"][["constructorId", "constructorRef"]], on="constructorId", how="left", validate="many_to_one"
)
display(example[[
    "year", "raceId", "driverId", "driverRef", "constructorRef", "number",
    "grid", "positionText", "positionOrder", "points", "laps", "statusId",
]].head(12))

,year,raceId,driverId,driverRef,constructorRef,number,grid,positionText,positionOrder,points,laps,statusId
0,1978,540,229,ertl,ats,10,0,F,26,0.0,0,81
1,1978,540,229,ertl,ensign,23,0,F,29,0.0,0,97
2,1964,717,373,clark,lotus-climax,1,1,7,7,0.0,102,60
3,1964,717,373,clark,lotus-climax,2,1,R,12,0.0,54,98
4,1962,733,465,greene,lotus-climax,48,0,W,22,0.0,0,54
5,1962,733,465,greene,gilby,50,0,W,25,0.0,0,54
6,1961,742,475,moss,ferguson,26,20,D,19,0.0,56,2
7,1961,742,475,moss,lotus-climax,28,5,R,21,0.0,44,23
8,1961,745,418,gregory,lotus-climax,22,11,R,17,0.0,23,6
9,1961,745,418,gregory,lotus-climax,21,15,11,11,0.0,92,18


### 3.5 Qualifying data: first year and coverage per season

Two coverage measures per season:

- **races covered**: share of races with *any* qualifying rows;
- **entries covered**: share of `results` rows (driver–race entries) whose
  `(raceId, driverId)` also appears in `qualifying`. This is what matters for our grain:
  it is the share of model rows that would have qualifying features.

In [12]:
quali_race_ids = set(qualifying["raceId"].dropna())
first_quali_year = int(races.loc[races["raceId"].isin(quali_race_ids), "year"].min())
print("First season with qualifying data:", first_quali_year)

quali_pairs = qualifying[["raceId", "driverId"]].drop_duplicates().assign(has_quali=True)
entries = with_year(results[["raceId", "driverId"]]).merge(
    quali_pairs, on=["raceId", "driverId"], how="left", validate="many_to_one"
)
assert len(entries) == len(results), "join changed the row count"
entries["has_quali"] = entries["has_quali"].fillna(False).astype(bool)

races_with_results = set(results["raceId"])
season = races.assign(
    has_results=races["raceId"].isin(races_with_results),
    has_quali=races["raceId"].isin(quali_race_ids),
)
quali_cov = pd.DataFrame({
    "n_races": season.groupby("year").size(),
    "n_races_with_results": season.groupby("year")["has_results"].sum(),
    "n_races_with_quali": season.groupby("year")["has_quali"].sum(),
    "n_results_rows": entries.groupby("year").size(),
    "n_results_rows_with_quali": entries.groupby("year")["has_quali"].sum(),
}).fillna(0).astype(int)
quali_cov["pct_races_covered"] = (100 * quali_cov["n_races_with_quali"] / quali_cov["n_races"]).round(1)
quali_cov["pct_entries_covered"] = (
    100 * quali_cov["n_results_rows_with_quali"] / quali_cov["n_results_rows"].replace(0, np.nan)
).round(1)
display(quali_cov)

First season with qualifying data: 1994


,n_races,n_races_with_results,n_races_with_quali,n_results_rows,n_results_rows_with_quali,pct_races_covered,pct_entries_covered
year,,,,,,,
1950,7,7,0,160,0,0.0,0.0
1951,8,8,0,179,0,0.0,0.0
1952,8,8,0,215,0,0.0,0.0
1953,9,9,0,246,0,0.0,0.0
1954,9,9,0,230,0,0.0,0.0
1955,7,7,0,180,0,0.0,0.0
1956,8,8,0,190,0,0.0,0.0
1957,8,8,0,171,0,0.0,0.0
1958,11,11,0,241,0,0.0,0.0


### 3.6 `grid = 0` by decade

`grid = 0` has no single documented meaning (e.g. pit-lane start vs. did not start).
We only count it here; how to treat it is an open decision.

In [13]:
grid = with_year(results[["raceId", "grid"]])
grid["decade"] = (grid["year"] // 10 * 10).astype(int)
grid_by_decade = grid.groupby("decade").agg(
    n_results_rows=("grid", "size"),
    n_grid_null=("grid", lambda s: int(s.isna().sum())),
    n_grid_zero=("grid", lambda s: int((s == 0).sum())),
)
grid_by_decade["pct_grid_zero"] = (100 * grid_by_decade["n_grid_zero"] / grid_by_decade["n_results_rows"]).round(2)
display(grid_by_decade)

,n_results_rows,n_grid_null,n_grid_zero,pct_grid_zero
decade,,,,
1950,2007,0,68,3.39
1960,2113,0,198,9.37
1970,3807,0,351,9.22
1980,4503,0,571,12.68
1990,4250,0,357,8.4
2000,3643,0,8,0.22
2010,4297,0,28,0.65
2020,2139,0,57,2.66


### 3.7 Findings → open decisions

*To be filled in after reviewing the outputs above.*

| Finding | Evidence (section) | Open decision | Proposed rule | Status |
|---|---|---|---|---|
| … | 3.4 | Shared-drive rule | … | Pending |
| … | 3.5 | Missing-qualifying strategy / training window | … | Pending |
| … | 3.6 | Meaning of `grid = 0` | … | Pending |